In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound, AutoScheme, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from torch import nn
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB")
        print(f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n")
else:
    print("No CUDA-compatible GPUs found.")

PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
Total GPUs Detected: 1

--- GPU 0 ---
  Name: NVIDIA B300 SXM6 AC
  Compute Capability: 10.3
  Total VRAM: 267.68 GB
  Allocated VRAM: 0.00 GB
  Free VRAM: 267.68 GB



In [4]:
notebook_login()

In [5]:
MODEL_ID = "Qwen/Qwen3.5-4B"
HF_USER = "Vishva007"               # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound-4b"
LOCAL_PATH = "./local_model-4b"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 14 files:   0%|                                | 0/14 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 5.33GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 9.32GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 9.32GB            
Reconstructing (incomplete total...):   0%|       | 11.5kB / 9.32GB            

Fetching 14 files:   7%|█▋                      | 1/14 [00:00<00:03,  3.32it/s]
Reconstructing (incomplete total...):   0%|       | 14.7kB / 9.32GB            
Reconstructing (incomplete total...):   0%|       | 16.3kB / 9.33GB            
Reconstructing (incomplete total...):   0%|   

In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [8]:
def get_qwen3_5_shared_layers(
    model: nn.Module,
    skip_vision_towers: bool = True,
) -> list[list[str]]:
    """Extracts all fused kernel layer groups that must share identical bit-widths

    to prevent vLLM loader assertion errors (AssertionError in load_merged_column_weight).
    """
    FUSION_RULES = [
        # Standard MLP: gate_proj and up_proj are fused
        ("mlp", ("gate_proj", "up_proj")),
        # Full Self-Attention: Q, K, V projections are fused
        ("self_attn", ("q_proj", "k_proj", "v_proj")),
        ("attn", ("q_proj", "k_proj", "v_proj")),
        # Gated DeltaNet Linear Attention: All 4 in-projections are merged together
        ("linear_attn", ("in_proj_qkv", "in_proj_z", "in_proj_b", "in_proj_a")),
    ]

    all_modules = dict(model.named_modules())
    shared_layers: list[list[str]] = []
    processed_parents: set[str] = set()

    for full_name, module in all_modules.items():
        if not isinstance(module, nn.Linear):
            continue

        if skip_vision_towers and any(p in full_name for p in ("visual.", "merger.")):
            continue

        parent_name, leaf_name = full_name.rsplit(".", 1) if "." in full_name else ("", full_name)
        parent_type = parent_name.rsplit(".", 1)[-1] if parent_name else ""

        if parent_name and parent_name not in processed_parents:
            # Check attention, linear attention, and MLP patterns
            for target_parent, siblings in FUSION_RULES:
                if parent_type == target_parent and leaf_name in siblings:
                    found_siblings = [
                        f"{parent_name}.{sib}"
                        for sib in siblings
                        if f"{parent_name}.{sib}" in all_modules
                    ]
                    if len(found_siblings) >= 2:
                        shared_layers.append(found_siblings)
                        processed_parents.add(parent_name)
                    break

            # Handle MoE expert layers if running a MoE variant
            if "experts" in parent_name and leaf_name in ("gate_proj", "up_proj"):
                found_siblings = [
                    f"{parent_name}.{sib}"
                    for sib in ("gate_proj", "up_proj")
                    if f"{parent_name}.{sib}" in all_modules
                ]
                if len(found_siblings) >= 2:
                    shared_layers.append(found_siblings)
                    processed_parents.add(parent_name)

    return shared_layers

In [9]:
shared_layers = get_qwen3_5_shared_layers(model, skip_vision_towers=True)

print(f"Auto-detected {len(shared_layers)} fused projection groups for vLLM.")

Auto-detected 64 fused projection groups for vLLM.


In [10]:

# Configure AutoScheme for dynamic mixed-precision layer allocation
mixed_scheme = AutoScheme(
    avg_bits=3.5,                                           # Target average bit-width across all quantized layers in the model
    options=["W2A16G32", "W3A16G32", "W4A16G32", "BF16"],   # Candidate precision schemes evaluated by the dynamic programming solver
    ignore_scale_zp_bits=True,                              # Excludes metadata overhead (scales and zero-points) from the average bit calculation
    shared_layers=shared_layers,                            # Enforces identical bit-widths on fused kernels (MLP gate/up, QKV) to prevent vLLM loader assertions
    nsamples=512,                                           # Total calibration dataset samples to maintain quality and avoid overfitting
    seqlen=4096,                                            # Calibration context length; captures long-range dependency activations
    batch_size=2,                                           # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
)

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,                                 # Optimization iterations per block; high accuracy (Production grade)
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme=mixed_scheme,                        # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    group_size=32,                              # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                                   # Symmetric quantization; enables faster compute and broader engine compatibility
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,                  # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    ignore_layers="mtp,visual,lm_head",         # Excludes MTP, and visual modules from quantization (Add linear_attn if needed and lm_head is skipped automatically due to tied weights)
    layer_config={
        "mtp": {
            "data_type": "bfloat16"
        },  # Keeps the MTP module in BF16 precision
        "mtp.fc": {
            "data_type": "bfloat16"
        },  # Keeps the MTP fully connected layer in BF16 precision
        "lm_head": {
            "data_type": "bfloat16"
        },  # Keeps head in BF16; Use at least 6-bit quantization, or at least 4-bit {"bits": 6} or {"bits": 4} if resources are limited.
    },  # Preserves unquantized precision for sensitive output and auxiliary layers
)

2026-11-07 03:22:55 WARNING autoround.py L499: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-11-07 03:22:55 WARNING autoround.py L499: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [11]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # - 'llm_compressor': Compressed-tensors schema (vLLM / Neural Magic)
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

2026-11-07 03:22:58 INFO base.py L812: AutoScheme on multimodal LLM: scoring the language tower only with text-only calibration (multimodal dataloader will be used as a fallback if needed).
2026-11-07 03:22:58 WARNING base.py L841: Models with mixed data_types cannot yet be exported to real formats except GGUF. Please save the model using the `fake` format for now.
2026-11-07 03:22:58 INFO layer_config_resolver.py L408: Ignored layers: lm_head, lm_head, model.visual.blocks.[0-23].attn.proj, model.visual.blocks.[0-23].attn.qkv, model.visual.blocks.[0-23].mlp.linear_fc1, model.visual.blocks.[0-23].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp, visual
2026-11-07 03:22:58 WARNING layer_config_resolver.py L61: reset `quant_lm_head` to false as quantizing lm_head with tied weights has not been supported currently
2026-11-07 03:22:58 INFO layer_config_resolver.py L408: Ignored layers: lm_head, lm_head, model.visual.blocks.[0-23].attn.proj, model.visual.bl

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [12]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Uploaded or Updated model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [13]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [14]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-4b-w2g32"),
        f"{base_name}-3.5Bits-LLM-Compressor",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound-4b/local_model-4b-w2g32 to Vishva007/Qwen3.5-4B-3.5Bits-LLM-Compressor...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-4B-3.5Bits-LLM-Compressor
